### Mission 2 — Pipeline et baseline (sans fuite de données)

Règle absolue : le split train/test précède toute transformation apprise
(imputation, encodage, scaling). Toutes ces transformations vivent dans un
`Pipeline` scikit-learn, `fit` sur le train uniquement.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = pd.read_csv("../data/Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

X = df.drop(columns=["customerID", "Churn"])
y = (df["Churn"] == "Yes").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train :", X_train.shape, "- Taux de churn :", y_train.mean().round(3))
print("Test  :", X_test.shape, "- Taux de churn :", y_test.mean().round(3))

Train : (5634, 19) - Taux de churn : 0.265
Test  : (1409, 19) - Taux de churn : 0.265


### Imputation conditionnelle de `TotalCharges`

Règle métier plutôt qu'une simple médiane : 0€ pour un client nouveau
(tenure=0), estimation tenure×MonthlyCharges pour un client existant avec
une valeur manquante (cas non observé ici mais possible en production).

In [2]:
from sklearn.base import BaseEstimator, TransformerMixin

class TotalChargesImputer(BaseEstimator, TransformerMixin):
    """Impute TotalCharges manquant selon une règle métier :
    - Client nouveau (tenure=0)   -> 0€ (n'a encore rien payé, c'est un fait)
    - Client existant (tenure>0)  -> tenure x MonthlyCharges (estimation
      spécifique à CE client, plus fiable qu'une médiane globale)
    Ne rien "apprendre" du train : règle déterministe appliquée ligne par ligne,
    donc pas de risque de fuite.
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        missing = X["TotalCharges"].isna()
        is_new = missing & (X["tenure"] == 0)
        is_existing = missing & (X["tenure"] > 0)
        X.loc[is_new, "TotalCharges"] = 0
        X.loc[is_existing, "TotalCharges"] = (
            X.loc[is_existing, "tenure"] * X.loc[is_existing, "MonthlyCharges"]
        )
        return X

# Test rapide sur le train
fixer = TotalChargesImputer()
X_train_fixed = fixer.transform(X_train)
print("NA restants après règle métier :", X_train_fixed["TotalCharges"].isna().sum())

NA restants après règle métier : 0


### Moyenne ou médiane pour l'imputation ?

Comparaison moyenne/médiane et calcul du skewness (asymétrie) pour
justifier le choix de la médiane comme stratégie d'imputation par défaut.

In [3]:
for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    mean_val = X_train[col].mean()
    median_val = X_train[col].median()
    skew_val = X_train[col].skew()
    print(f"{col}: moyenne={mean_val:.1f}, médiane={median_val:.1f}, skewness={skew_val:.2f}")

tenure: moyenne=32.5, médiane=29.0, skewness=0.24
MonthlyCharges: moyenne=64.9, médiane=70.5, skewness=-0.22
TotalCharges: moyenne=2302.6, médiane=1398.1, skewness=0.95


## 2. Pipeline de préparation (ColumnTransformer)

Sous-pipeline numérique : imputation médiane (filet de sécurité, justifiée
par le skewness ci-dessus) + StandardScaler.
Sous-pipeline catégoriel : imputation mode + OneHotEncoder.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X_train.columns if c not in num_cols]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols),
])

print("Colonnes numériques :", num_cols)
print("Colonnes catégorielles :", cat_cols)

Colonnes numériques : ['tenure', 'MonthlyCharges', 'TotalCharges']
Colonnes catégorielles : ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 3. Baseline — régression logistique

Pipeline complet (imputation métier + ColumnTransformer + régression
logistique), évalué en validation croisée 5 plis sur le train uniquement.
C'est le score de référence à battre pour la suite du projet.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

baseline_pipeline = Pipeline([
    ("total_charges_fix", TotalChargesImputer()),
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores_recall = cross_val_score(baseline_pipeline, X_train, y_train, cv=cv, scoring="recall")
scores_f1 = cross_val_score(baseline_pipeline, X_train, y_train, cv=cv, scoring="f1")
scores_acc = cross_val_score(baseline_pipeline, X_train, y_train, cv=cv, scoring="accuracy")

print(f"Rappel  : {scores_recall.mean():.3f} ± {scores_recall.std():.3f}")
print(f"F1      : {scores_f1.mean():.3f} ± {scores_f1.std():.3f}")
print(f"Accuracy: {scores_acc.mean():.3f} ± {scores_acc.std():.3f}")

Rappel  : 0.544 ± 0.041
F1      : 0.592 ± 0.030
Accuracy: 0.802 ± 0.012


## 4. Feature engineering

3 nouvelles features : nombre de services souscrits, tranche d'ancienneté,
charges par mois d'ancienneté. Transformations ligne par ligne (pas
d'apprentissage), donc intégrables au pipeline sans risque de fuite.

In [6]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Ajoute 3 features dérivées, sans rien apprendre du train (transformations
    ligne par ligne uniquement -> pas de risque de fuite)."""

    service_cols = [
        "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
        "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
        "StreamingMovies",
    ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["n_services"] = X[self.service_cols].apply(
            lambda col: col.isin(["Yes"]) if col.name != "InternetService"
            else ~col.isin(["No"]), axis=0
        ).sum(axis=1)

        X["tenure_bucket"] = pd.cut(
            X["tenure"], bins=[-1, 12, 36, 100],
            labels=["nouveau_0_12m", "etabli_12_36m", "fidele_36m_plus"]
        ).astype(str)

        X["charges_per_tenure"] = X["TotalCharges"] / (X["tenure"] + 1)
        return X

fe = FeatureEngineer()
X_train_fe = fe.transform(TotalChargesImputer().transform(X_train))
print(X_train_fe[["n_services", "tenure_bucket", "charges_per_tenure"]].describe(include="all"))

         n_services    tenure_bucket  charges_per_tenure
count   5634.000000             5634         5634.000000
unique          NaN                3                 NaN
top             NaN  fidele_36m_plus                 NaN
freq            NaN             2410                 NaN
mean       4.167554              NaN           59.206762
std        2.320947              NaN           30.627621
min        1.000000              NaN            0.000000
25%        2.000000              NaN           26.274464
50%        4.000000              NaN           61.259214
75%        6.000000              NaN           85.244497
max        9.000000              NaN          118.969863


## 5. Baseline vs Feature Engineering — gain chiffré

In [7]:
num_cols_fe = num_cols + ["n_services", "charges_per_tenure"]
cat_cols_fe = cat_cols + ["tenure_bucket"]

preprocessor_fe = ColumnTransformer([
    ("num", numeric_pipeline, num_cols_fe),
    ("cat", categorical_pipeline, cat_cols_fe),
])

fe_pipeline = Pipeline([
    ("total_charges_fix", TotalChargesImputer()),
    ("feature_engineer", FeatureEngineer()),
    ("preprocessor", preprocessor_fe),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

scores_recall_fe = cross_val_score(fe_pipeline, X_train, y_train, cv=cv, scoring="recall")
scores_f1_fe = cross_val_score(fe_pipeline, X_train, y_train, cv=cv, scoring="f1")
scores_acc_fe = cross_val_score(fe_pipeline, X_train, y_train, cv=cv, scoring="accuracy")

print("=== Baseline (sans feature engineering) ===")
print(f"Rappel  : {scores_recall.mean():.3f} ± {scores_recall.std():.3f}")
print(f"F1      : {scores_f1.mean():.3f} ± {scores_f1.std():.3f}")
print(f"Accuracy: {scores_acc.mean():.3f} ± {scores_acc.std():.3f}")

print("\n=== Avec feature engineering ===")
print(f"Rappel  : {scores_recall_fe.mean():.3f} ± {scores_recall_fe.std():.3f}")
print(f"F1      : {scores_f1_fe.mean():.3f} ± {scores_f1_fe.std():.3f}")
print(f"Accuracy: {scores_acc_fe.mean():.3f} ± {scores_acc_fe.std():.3f}")

print(f"\nGain rappel : {(scores_recall_fe.mean() - scores_recall.mean())*100:+.1f} points")
print(f"Gain F1     : {(scores_f1_fe.mean() - scores_f1.mean())*100:+.1f} points")

=== Baseline (sans feature engineering) ===
Rappel  : 0.544 ± 0.041
F1      : 0.592 ± 0.030
Accuracy: 0.802 ± 0.012

=== Avec feature engineering ===
Rappel  : 0.522 ± 0.033
F1      : 0.588 ± 0.023
Accuracy: 0.806 ± 0.007

Gain rappel : -2.1 points
Gain F1     : -0.5 points


### Test individuel de chaque feature (ablation)

Le gain combiné des 3 features était négatif — on teste chacune séparément
pour identifier laquelle pose problème, plutôt que de toutes les rejeter.

In [8]:
def build_pipeline(add_n_services=False, add_tenure_bucket=False, add_charges_per_tenure=False):
    class SelectiveFE(BaseEstimator, TransformerMixin):
        def fit(self, X, y=None):
            return self
        def transform(self, X):
            X = X.copy()
            if add_n_services or add_charges_per_tenure:
                pass
            if add_n_services:
                X["n_services"] = X[FeatureEngineer.service_cols].apply(
                    lambda col: col.isin(["Yes"]) if col.name != "InternetService"
                    else ~col.isin(["No"]), axis=0
                ).sum(axis=1)
            if add_tenure_bucket:
                X["tenure_bucket"] = pd.cut(
                    X["tenure"], bins=[-1, 12, 36, 100],
                    labels=["nouveau_0_12m", "etabli_12_36m", "fidele_36m_plus"]
                ).astype(str)
            if add_charges_per_tenure:
                X["charges_per_tenure"] = X["TotalCharges"] / (X["tenure"] + 1)
            return X

    nc = num_cols + (["n_services"] if add_n_services else []) + (["charges_per_tenure"] if add_charges_per_tenure else [])
    cc = cat_cols + (["tenure_bucket"] if add_tenure_bucket else [])
    prep = ColumnTransformer([("num", numeric_pipeline, nc), ("cat", categorical_pipeline, cc)])

    return Pipeline([
        ("total_charges_fix", TotalChargesImputer()),
        ("selective_fe", SelectiveFE()),
        ("preprocessor", prep),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])

configs = {
    "baseline (aucune)": dict(),
    "+ n_services seule": dict(add_n_services=True),
    "+ tenure_bucket seule": dict(add_tenure_bucket=True),
    "+ charges_per_tenure seule": dict(add_charges_per_tenure=True),
}

for name, kwargs in configs.items():
    pipe = build_pipeline(**kwargs)
    r = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="recall")
    print(f"{name:30s} rappel = {r.mean():.3f} ± {r.std():.3f}")

baseline (aucune)              rappel = 0.544 ± 0.041
+ n_services seule             rappel = 0.543 ± 0.041
+ tenure_bucket seule          rappel = 0.532 ± 0.030
+ charges_per_tenure seule     rappel = 0.532 ± 0.038


## 6. Conclusion — Pipeline final de la Mission 2

| Configuration | Rappel (CV) |
|---|---|
| **Baseline (retenue)** | **0,544 ± 0,041** |
| + n_services seule | 0,543 ± 0,041 (neutre) |
| + tenure_bucket seule | 0,532 ± 0,030 (dégrade) |
| + charges_per_tenure seule | 0,532 ± 0,038 (dégrade) |

Aucune des 3 features engineered n'apporte de gain réel sur le rappel —
elles sont donc **rejetées** (rasoir d'Occam). Le pipeline retenu pour la
suite du projet est `baseline_pipeline` : `TotalChargesImputer` (règle
métier) → `ColumnTransformer` (imputation médiane/mode + scaling/OHE) →
modèle. Ce choix est documenté et justifié plutôt qu'arbitraire — la
Mission 3 comparera d'autres familles de modèles sur cette même base.